# 04 · Recuperación semántica (Sentence-Transformers)

Embeddings con `all-MiniLM-L6-v2` + búsqueda híbrida BM25 ⊕ semántica.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src import config
from src.retrieval import BM25Retriever, SemanticRetriever
from src.retrieval.semantic_retriever import HybridRetriever

df = pd.read_parquet(config.CORPUS_PATH).head(5000)  # subset para rapidez
print('Subset:', len(df))

In [ ]:
sem = SemanticRetriever().fit(df['text'].tolist())
sem.save(config.PROCESSED_DIR / 'embeddings_semantic')

In [ ]:
bm25 = BM25Retriever().fit(df['clean_text'].tolist())
hybrid = HybridRetriever(bm25, sem, alpha=0.5)

queries = [
    'covid vaccine side effects in children',
    'climate change is a hoax',
    'election was stolen and rigged',
]
for q in queries:
    print(f'\n=== {q!r} ===')
    for retr in (bm25, sem, hybrid):
        idx, sc = retr.search(q, top_k=3)
        print(f'  {retr.name:>8} ->', [f"{df.iloc[i]['title'][:60]} ({sc[k]:.2f})" for k,i in enumerate(idx)])